Install packages on colab.

In [1]:
# Install uv for fast package management
!pip install uv

# Install required libraries
!uv pip install torch torchvision torchaudio wandb fiftyone huggingface_hub scikit-learn matplotlib tqdm --system

Using Python 3.11.11 environment at: C:\Users\Philipp\miniconda3
Resolved 134 packages in 1.73s
 Downloaded wandb
Prepared 3 packages in 3.23s
Installed 11 packages in 25.01s
 + gitdb==4.0.12
 + gitpython==3.1.45
 + mpmath==1.3.0
 + protobuf==6.33.2
 + sentry-sdk==2.48.0
 + smmap==5.0.2
 + sympy==1.14.0
 + torch==2.9.1
 + torchaudio==2.9.1
 + torchvision==0.24.1
 + wandb==0.23.1


Import and initialize seeds.

In [28]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import Adam
import torch.nn.functional as F
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
import fiftyone as fo
import fiftyone.utils.huggingface as fouh
import wandb
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt
from PIL import Image


# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


Login to huggingface and wandb using API Keys.

In [4]:
!hf auth login

User is already logged in.


In [12]:
!wandb login

wandb: Currently logged in as: hpi-philipp-kolbe (na_mst_2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Load Dataset
Load the dataset from Hugging Face (created in Task 2).

In [3]:
# Configuration
HF_USERNAME = "philippkolbe" 
SUBSET_NAME = "cilp_assessment_subset"
HF_DATASET_REPO = f"{HF_USERNAME}/{SUBSET_NAME}"

# Load dataset from Hugging Face
print(f"Loading dataset from {HF_DATASET_REPO}...")
# Check if dataset exists locally
if fo.dataset_exists(SUBSET_NAME):
    print(f"Dataset '{SUBSET_NAME}' found locally. Loading...")
    dataset = fo.load_dataset(SUBSET_NAME)
else:
    print(f"Dataset '{SUBSET_NAME}' not found locally. Loading from Hugging Face ({HF_DATASET_REPO})...")
    try:
        # Load from Hub and name it SUBSET_NAME to persist it locally with the right name
        dataset = fouh.load_from_hub(HF_DATASET_REPO, name=SUBSET_NAME)
        print("Dataset loaded successfully from Hub.")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Make sure you are logged in to Hugging Face (huggingface-cli login).")

# Print summary
print(dataset)

Loading dataset from philippkolbe/cilp_assessment_subset...


fiftyone.yml:   0%|          | 0.00/107 [00:00<?, ?B/s]

Loading dataset


c:\Users\Philipp\2_uni\wise2526\AHOCV\Applied-Hands-On-Computer-Vision\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Philipp\.cache\huggingface\hub\datasets--philippkolbe--cilp_assessment_subset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Importing samples...
 100% |███████████████| 2250/2250 [123.8ms elapsed, 0s remaining, 18.2K samples/s]    


100%|██████████| 23/23 [05:06<00:00, 13.35s/it]

Dataset loaded successfully.
Name:        philippkolbe/cilp_assessment_subset
Media type:  group
Group slice: rgb
Num groups:  750
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    group:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    raw_filepath:     fiftyone.core.fields.StringField


Load ground truths from the dataset to load their azimuth and zenith files.

In [17]:
ground_truths = set(dataset.values("ground_truth.label"))
ground_truths

{'cubes', 'spheres'}

For this to work you will need to get the azimuth and zenith files from the original dataset (`data/assessment/cubes|spheres/azimuth|zenith.npy`). We assume that they are the same for both cubes and spheres.

In [24]:
azimuths = {}
# for each ground_truth label in dataset
for gt in ground_truths:
    azimuth = np.load(f"data/assessment/{gt}/azimuth.npy")
    azimuths[gt] = azimuth
    print(gt, azimuth.shape)

azimuth = next(iter(azimuths.values())) # just choose any because they should all be the same

azimuths

cubes (64,)
spheres (64,)


{'cubes': array([-0.27925265, -0.27052593, -0.26179934, -0.25307274, -0.24434602,
        -0.23561943, -0.22689271, -0.21816611, -0.20943952, -0.2007128 ,
        -0.1919862 , -0.18325949, -0.17453289, -0.1658063 , -0.15707958,
        -0.14835298, -0.13962626, -0.13089967, -0.12217295, -0.11344635,
        -0.10471976, -0.09599304, -0.08726645, -0.07853973, -0.06981313,
        -0.06108654, -0.05235982, -0.04363322, -0.03490651, -0.02617991,
        -0.01745319, -0.0087266 ,  0.        ,  0.00872672,  0.01745331,
         0.02618003,  0.03490663,  0.04363322,  0.05235994,  0.06108654,
         0.06981325,  0.07853985,  0.08726656,  0.09599316,  0.10471976,
         0.11344647,  0.12217307,  0.13089979,  0.13962638,  0.14835298,
         0.1570797 ,  0.1658063 ,  0.17453301,  0.1832596 ,  0.1919862 ,
         0.20071292,  0.20943952,  0.21816623,  0.22689283,  0.23561954,
         0.24434614,  0.25307274,  0.26179945,  0.27052605], dtype=float32),
 'spheres': array([-0.27925265, -0.270

In [25]:
zeniths = {}
# for each ground_truth label in dataset
for gt in ground_truths:
    zenith = np.load(f"data/assessment/{gt}/zenith.npy")
    zeniths[gt] = zenith
    print(gt, zenith.shape)

zenith = next(iter(zeniths.values())) # just choose any because they should all be the same

zeniths

cubes (64,)
spheres (64,)


{'cubes': array([-0.27925268, -0.27052602, -0.2617994 , -0.25307274, -0.2443461 ,
        -0.23561946, -0.2268928 , -0.21816616, -0.20943952, -0.20071286,
        -0.19198622, -0.18325958, -0.17453292, -0.16580628, -0.15707964,
        -0.14835298, -0.13962634, -0.1308997 , -0.12217305, -0.1134464 ,
        -0.10471976, -0.09599311, -0.08726646, -0.07853982, -0.06981317,
        -0.06108652, -0.05235988, -0.04363323, -0.03490658, -0.02617994,
        -0.01745329, -0.00872665,  0.        ,  0.00872665,  0.01745329,
         0.02617994,  0.03490658,  0.04363323,  0.05235988,  0.06108652,
         0.06981317,  0.07853982,  0.08726646,  0.09599311,  0.10471976,
         0.1134464 ,  0.12217305,  0.1308997 ,  0.13962634,  0.14835298,
         0.15707964,  0.16580628,  0.17453292,  0.18325958,  0.19198622,
         0.20071286,  0.20943952,  0.21816616,  0.2268928 ,  0.23561946,
         0.2443461 ,  0.25307274,  0.2617994 ,  0.27052602], dtype=float32),
 'spheres': array([-0.27925268, -0.270

## 2. Prepare Data Loaders
Create a PyTorch Dataset class to handle RGB and LiDAR pairs.

In [37]:
def get_torch_xyza(lidar_depth, azimuth, zenith):
    x = lidar_depth * torch.sin(-azimuth[:, None]) * torch.cos(-zenith[None, :])
    y = lidar_depth * torch.cos(-azimuth[:, None]) * torch.cos(-zenith[None, :])
    z = lidar_depth * torch.sin(-zenith[None, :])
    a = torch.where(lidar_depth < 50.0, torch.ones_like(lidar_depth), torch.zeros_like(lidar_depth))
    xyza = torch.stack((x, y, z, a))
    return xyza

Transforms for scaling and converting to tensors.

In [36]:
IMG_SIZE = 64
BATCH_SIZE = 32

img_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),  # Scales data into [0,1]
])

Define data loaders for training and validation sets.

In [ ]:
class MultimodalDataset(Dataset):
    """Custom Dataset for loading RGB images and LiDAR depth maps from FiftyOne dataset."""
    def __init__(self, fiftyone_dataset, split, azimuth, zenith, transform):
        """Initialize the dataset by filtering samples based on the split and loading RGB and LiDAR data.
        
        Args:
            fiftyone_dataset (fiftyone.core.dataset.Dataset): The FiftyOne dataset containing the samples.
            split (str): The split tag to filter samples (e.g., "train", "val", "test").
            azimuth (np.ndarray): The azimuth angles corresponding to the LiDAR data.
            zenith (np.ndarray): The zenith angles corresponding to the LiDAR data.
            transform (callable): Transformations to apply to the RGB images.
        """
        self.transform = transform
        self.samples = []
        
        # Filter by split tag
        view = fiftyone_dataset.match_tags(split)

        self.azimuth = torch.from_numpy(azimuth).to(device)
        self.zenith = torch.from_numpy(zenith).to(device)

        # Get RGB and LiDAR slices
        rgb_view = view.select_group_slices("rgb")
        lidar_view = view.select_group_slices("lidar")
        
        # Create a map of group_id -> lidar_filepath
        lidar_map = {s.group.id: s.filepath for s in lidar_view}
        
        for s in rgb_view:
            if s.group.id in lidar_map:
                rgb_filepath = s.filepath
                rgb_img = Image.open(rgb_filepath)
                rgb_img = self.transform(rgb_img).to(device)

                lidar_filepath = lidar_map[s.group.id]
                lidar_depth = np.load(lidar_filepath)
                lidar_depth = torch.from_numpy(lidar_depth).to(torch.float32).to(device)

                self.samples.append({
                    "rgb": rgb_img,
                    "lidar": lidar_depth,
                    "label": 0 if s.ground_truth.label == "cubes" else 1 # cubes=0, spheres=1
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        rbg_img = item["rgb"]
        lidar_depth = item["lidar"]
        lidar_xyza = get_torch_xyza(lidar_depth, self.azimuth, self.zenith)

        label_tensor = torch.tensor(item["label"], dtype=torch.float32)

        return rbg_img, lidar_xyza, label_tensor

# Create Datasets
train_dataset = MultimodalDataset(dataset, "train", azimuth, zenith, img_transforms)
val_dataset = MultimodalDataset(dataset, "val", azimuth, zenith, img_transforms)
test_dataset = MultimodalDataset(dataset, "test", azimuth, zenith, img_transforms)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 505
Val samples: 123
Test samples: 122


## 3. Define Models

### 3.1 Late Fusion Architecture
Process modalities separately and concatenate final embeddings.

**Advantages:**
- **Modularity:** Each modality can use a specialized encoder (e.g., CNN for images, PointNet for LiDAR points) without forcing them to share structure early on.
- **Simplicity:** Easy to implement; just train separate networks and fuse at the end.
- **Pre-training:** Easier to use off-the-shelf pre-trained backbones for each modality.

**Limitations:**
- **Late Interaction:** The network cannot learn low-level geometric correlations between modalities (e.g., an edge in RGB corresponding to a depth discontinuity in LiDAR) until the very last layers.

In [ ]:
class LateFusionModel(nn.Module):
    """Late Fusion Model for multimodal classification using RGB images and LiDAR data."""
    def __init__(self):
        super(LateFusionModel, self).__init__()
        
        # RGB Encoder
        self.rgb_encoder = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 64 -> 32
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 32 -> 16
            nn.Flatten(),
            nn.Linear(100 * 16 * 16, 100),
            nn.ReLU()
        )
        
        # LiDAR Encoder
        self.lidar_encoder = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1), # LiDAR has 4 channels (XYZA)
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(100 * 16 * 16, 100),
            nn.ReLU()
        )
        
        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(200, 100), # 100 RGB + 100 LiDAR
            nn.ReLU(),
            nn.Linear(100, 1)
        )
        
    def forward(self, rgb, lidar):
        rgb_emb = self.rgb_encoder(rgb)
        lidar_emb = self.lidar_encoder(lidar)
        
        combined = torch.cat((rgb_emb, lidar_emb), dim=1)
        output = self.classifier(combined)
        return output

### 3.2 Intermediate Fusion Architecture
Combine feature maps at an intermediate layer.

**Advantages:**
- **Feature Interaction:** Allows the network to learn cross-modal features (e.g., using depth info to disambiguate texture in RGB) at earlier stages.
- **Rich Representation:** Can capture joint patterns that are only visible when looking at both modalities simultaneously in spatial context.

**Limitations:**
- **Alignment:** Requires feature maps to be spatially aligned (same height/width) at the fusion point.
- **Complexity:** Harder to design; need to decide *where* to fuse (early, mid, deep) and *how* (add, concat, multiply).

In [ ]:
class IntermediateFusionModel(nn.Module):
    """
    Intermediate Fusion Architecture.
    
    Combines feature maps from RGB and LiDAR modalities at an intermediate layer
    using Concatenation, Addition, or Hadamard Product (Multiplication).
    
    Args:
        fusion_type (str): Strategy to combine features. Options: 'concat', 'add', 'multiply'.
    """
    def __init__(self, fusion_type='concat'):
        super(IntermediateFusionModel, self).__init__()
        self.fusion_type = fusion_type
        
        # Early Convolutions (RGB)
        self.rgb_conv = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output: 100 x 16 x 16
        )
        
        # Early Convolutions (LiDAR)
        self.lidar_conv = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1), # LiDAR has 4 channels (XYZA)
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output: 100 x 16 x 16
        )
        
        # Determine input channels for shared layers
        if fusion_type == 'concat':
            shared_in_channels = 200
        else: # add or multiply
            shared_in_channels = 100
            
        # Shared Layers
        self.shared_layers = nn.Sequential(
            nn.Conv2d(shared_in_channels, 200, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 16 -> 8
            nn.Flatten(),
            nn.Linear(200 * 8 * 8, 100),
            nn.ReLU(),
            nn.Linear(100, 1)
        )
        
    def forward(self, rgb, lidar):
        x_rgb = self.rgb_conv(rgb)
        x_lidar = self.lidar_conv(lidar)
        
        if self.fusion_type == 'concat':
            combined = torch.cat((x_rgb, x_lidar), dim=1)
        elif self.fusion_type == 'add':
            combined = x_rgb + x_lidar
        elif self.fusion_type == 'multiply':
            combined = x_rgb * x_lidar
        else:
            raise ValueError(f"Unknown fusion type: {self.fusion_type}")
            
        output = self.shared_layers(combined)
        return output

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, config, entity="hpi-philipp-kolbe", project_name="cilp-extended-assessment"):
    """Train the given model with specified data loaders and configuration.
    
    Args:
        model (nn.Module): The neural network model to train.
        train_loader (DataLoader): DataLoader for the training set.
        val_loader (DataLoader): DataLoader for the validation set.
        config (dict): Configuration dictionary containing hyperparameters.
        entity (str): W&B entity name.
        project_name (str): W&B project name.
    """
    # Initialize W&B
    wandb.init(entity=entity, project=project_name, config=config, reinit=True)
    
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    best_val_loss = float('inf')
    best_val_loss_f1 = float('inf')
    
    print(f"Starting training for {config['architecture']} ({config['fusion_strategy']})...")
    
    for epoch in range(config["epochs"]):
        # Training
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []
        
        for rgb, lidar, labels in train_loader:
            rgb, lidar, labels = rgb.to(device), lidar.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(rgb, lidar).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy() > 0.5)
            train_targets.extend(labels.cpu().numpy())
            
        avg_train_loss = train_loss / len(train_loader)
        train_acc = accuracy_score(train_targets, train_preds)
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for rgb, lidar, labels in val_loader:
                rgb, lidar, labels = rgb.to(device), lidar.to(device), labels.to(device)
                outputs = model(rgb, lidar).squeeze()
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_preds.extend(torch.sigmoid(outputs).cpu().numpy() > 0.5)
                val_targets.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds)
        
        # Log to W&B
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_acc": train_acc,
            "val_loss": avg_val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        })
        
        print(f"Epoch {epoch+1}/{config['epochs']} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_loss_f1 = val_f1
        
    # Finish run
    wandb.finish()
    
    return {
        "val_loss": best_val_loss,
        "val_f1": best_val_loss_f1,
        "parameters": sum(p.numel() for p in model.parameters())
    }

## 5. Run Experiments

Now we can run experiments for both architectures and compare their performance on the validation set.

Initialize configurations for both models.

In [53]:
# Login to W&B
wandb.login()

results = {}

# Common Config
EPOCHS = 10
LR = 1e-3

Run training for Late Fusion Model.

In [54]:
# 1. Late Fusion
config_late = {
    "architecture": "Late Fusion",
    "fusion_strategy": "late",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_late = LateFusionModel()
results["Late Fusion"] = train_model(model_late, train_loader, val_loader, config_late)

Starting training for Late Fusion (late)...
Epoch 1/10 - Train Loss: 1.5227 - Val Loss: 0.6958 - Val Acc: 0.4715
Epoch 2/10 - Train Loss: 0.6961 - Val Loss: 0.6927 - Val Acc: 0.4715
Epoch 3/10 - Train Loss: 0.6900 - Val Loss: 0.6851 - Val Acc: 0.4715
Epoch 4/10 - Train Loss: 0.6476 - Val Loss: 0.5920 - Val Acc: 0.7073
Epoch 5/10 - Train Loss: 0.5431 - Val Loss: 0.5122 - Val Acc: 0.7398
Epoch 6/10 - Train Loss: 0.5273 - Val Loss: 0.4823 - Val Acc: 0.8049
Epoch 7/10 - Train Loss: 0.4906 - Val Loss: 0.4856 - Val Acc: 0.7805
Epoch 8/10 - Train Loss: 0.4553 - Val Loss: 0.4803 - Val Acc: 0.7886
Epoch 9/10 - Train Loss: 0.4085 - Val Loss: 0.4868 - Val Acc: 0.8049


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 10/10 - Train Loss: 0.3486 - Val Loss: 0.5136 - Val Acc: 0.7967


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▂▃▆▆▆▇▇█
train_loss,█▃▃▃▂▂▂▂▁▁
val_acc,▁▁▁▆▇█▇███
val_f1,▁▁▁▇██████
val_loss,███▅▂▁▁▁▁▂
epoch,10
train_acc,0.84356
train_loss,0.34856
val_acc,0.79675
val_f1,0.8227


Run training for Intermediate Fusion Model with Concatenation.

In [55]:
# 2. Intermediate Fusion (Concat)
config_concat = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "concat",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_concat = IntermediateFusionModel(fusion_type='concat')
results["Intermediate (Concat)"] = train_model(model_concat, train_loader, val_loader, config_concat)

Starting training for Intermediate Fusion (concat)...
Epoch 1/10 - Train Loss: 1.1881 - Val Loss: 0.6945 - Val Acc: 0.4715
Epoch 2/10 - Train Loss: 0.6953 - Val Loss: 0.6885 - Val Acc: 0.5285
Epoch 3/10 - Train Loss: 0.6957 - Val Loss: 0.7108 - Val Acc: 0.4715
Epoch 4/10 - Train Loss: 0.6941 - Val Loss: 0.7013 - Val Acc: 0.4715
Epoch 5/10 - Train Loss: 0.6890 - Val Loss: 0.7465 - Val Acc: 0.4715
Epoch 6/10 - Train Loss: 0.6505 - Val Loss: 0.6620 - Val Acc: 0.4797
Epoch 7/10 - Train Loss: 0.6058 - Val Loss: 0.5130 - Val Acc: 0.8130
Epoch 8/10 - Train Loss: 0.5271 - Val Loss: 0.4450 - Val Acc: 0.8211
Epoch 9/10 - Train Loss: 0.4752 - Val Loss: 0.4206 - Val Acc: 0.8293


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 10/10 - Train Loss: 0.4932 - Val Loss: 0.5261 - Val Acc: 0.7236


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▂▂▁▂▃▃▅▇██
train_loss,█▃▃▃▃▃▂▂▁▁
val_acc,▁▂▁▁▁▁███▆
val_f1,▁▇▁▁▁▂████
val_loss,▇▇▇▇█▆▃▂▁▃
epoch,10
train_acc,0.76238
train_loss,0.49316
val_acc,0.72358
val_f1,0.79268


Run training for Intermediate Fusion Model with Addition.

In [56]:
# 3. Intermediate Fusion (Add)
config_add = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "add",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_add = IntermediateFusionModel(fusion_type='add')
results["Intermediate (Add)"] = train_model(model_add, train_loader, val_loader, config_add)

Starting training for Intermediate Fusion (add)...
Epoch 1/10 - Train Loss: 1.3508 - Val Loss: 0.7115 - Val Acc: 0.4715
Epoch 2/10 - Train Loss: 0.6990 - Val Loss: 0.6942 - Val Acc: 0.4715
Epoch 3/10 - Train Loss: 0.6957 - Val Loss: 0.6938 - Val Acc: 0.4715
Epoch 4/10 - Train Loss: 0.6924 - Val Loss: 0.6957 - Val Acc: 0.4715
Epoch 5/10 - Train Loss: 0.6927 - Val Loss: 0.6947 - Val Acc: 0.4715
Epoch 6/10 - Train Loss: 0.6799 - Val Loss: 0.6888 - Val Acc: 0.4715
Epoch 7/10 - Train Loss: 0.6536 - Val Loss: 0.6184 - Val Acc: 0.4959
Epoch 8/10 - Train Loss: 0.5854 - Val Loss: 0.5016 - Val Acc: 0.7805
Epoch 9/10 - Train Loss: 0.5167 - Val Loss: 0.4508 - Val Acc: 0.7886


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 10/10 - Train Loss: 0.4829 - Val Loss: 0.4722 - Val Acc: 0.7886


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▁▂▂▂▁▂▆▇█
train_loss,█▃▃▃▃▃▂▂▁▁
val_acc,▁▁▁▁▁▁▂███
val_f1,▁▁▁▁▁▁▃███
val_loss,█████▇▆▂▁▂
epoch,10
train_acc,0.76832
train_loss,0.48291
val_acc,0.78862
val_f1,0.79365


Run training for Intermediate Fusion Model with Multiplication.

In [57]:

# 4. Intermediate Fusion (Multiply)
config_mult = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "multiply",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_mult = IntermediateFusionModel(fusion_type='multiply')
results["Intermediate (Multiply)"] = train_model(model_mult, train_loader, val_loader, config_mult)

Starting training for Intermediate Fusion (multiply)...
Epoch 1/10 - Train Loss: 0.8602 - Val Loss: 0.6946 - Val Acc: 0.4715
Epoch 2/10 - Train Loss: 0.6586 - Val Loss: 0.6162 - Val Acc: 0.6016
Epoch 3/10 - Train Loss: 0.5744 - Val Loss: 0.5330 - Val Acc: 0.7398
Epoch 4/10 - Train Loss: 0.4998 - Val Loss: 0.4316 - Val Acc: 0.8293
Epoch 5/10 - Train Loss: 0.4476 - Val Loss: 0.3869 - Val Acc: 0.8699
Epoch 6/10 - Train Loss: 0.4104 - Val Loss: 0.4288 - Val Acc: 0.8049
Epoch 7/10 - Train Loss: 0.3840 - Val Loss: 0.4003 - Val Acc: 0.8699
Epoch 8/10 - Train Loss: 0.3846 - Val Loss: 0.3018 - Val Acc: 0.8862
Epoch 9/10 - Train Loss: 0.4185 - Val Loss: 0.3962 - Val Acc: 0.8537


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 10/10 - Train Loss: 0.3847 - Val Loss: 0.2875 - Val Acc: 0.9106


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▁▅▆▇█████
train_loss,█▅▄▃▂▁▁▁▂▁
val_acc,▁▃▅▇▇▆▇█▇█
val_f1,▁▅▇██▇████
val_loss,█▇▅▃▃▃▃▁▃▁
epoch,10
train_acc,0.8297
train_loss,0.38472
val_acc,0.91057
val_f1,0.92086


In [58]:
results

{'Late Fusion': {'val_loss': 0.4802565276622772,
  'val_f1': 0.8194444444444444,
  'parameters': 5234301},
 'Intermediate (Concat)': {'val_loss': 0.4205538257956505,
  'val_f1': 0.8467153284671532,
  'parameters': 1734301},
 'Intermediate (Add)': {'val_loss': 0.4508018419146538,
  'val_f1': 0.8088235294117647,
  'parameters': 1554301},
 'Intermediate (Multiply)': {'val_loss': 0.2875341698527336,
  'val_f1': 0.920863309352518,
  'parameters': 1554301}}

## 6. Comparison Results

| Metric | Late Fusion | Intermediate (Concat) | Intermediate (Add) | Intermediate (Hadamard) |
|---|---:|---:|---:|---:|
| Validation Loss | 0.48 | 0.42 | 0.45 | 0.29 |
| F1 score | 0.82 | 0.85 | 0.81 | 0.92 |
| Parameters (count) | 5,234,301 | 1,734,301 | 1,554,301 | 1,554,301 |
| Runtime | 42s | 54s | 1m7s | 1m19s |

In [59]:
print(f"{'Architecture':<25} | {'Val Loss':<10} | {'F1 Score':<10} | {'Params':<10}")
print("-" * 65)
for name, res in results.items():
    print(f"{name:<25} | {res['val_loss']:.4f}     | {res['val_f1']:.4f}     | {res['parameters']}")

Architecture              | Val Loss   | F1 Score   | Params    
-----------------------------------------------------------------
Late Fusion               | 0.4803     | 0.8194     | 5234301
Intermediate (Concat)     | 0.4206     | 0.8467     | 1734301
Intermediate (Add)        | 0.4508     | 0.8088     | 1554301
Intermediate (Multiply)   | 0.2875     | 0.9209     | 1554301


![media/wandb_fusion_comparison.png](media/wandb_fusion_comparison.png)

## 7. Written Analysis of Results

Firstly, The experiments demonstrate a clear advantage for Intermediate Fusion, specifically using the Hadamard product, which achieved the highest F1 score of 0.92 and the lowest validation loss of 0.29. The Late Fusion model, even though having over 3x the parameters (5.2M vs 1.5M), performed worse with an F1 score of 0.82.

This shows that early feature interaction is important in this task. In Late Fusion, the ntwork processes RGB and LiDAR independently until the final classification layer. This prevents the model from learning low-level geometric correlations, e.g. aligning an edge in the RGB image with a depth difference in the LiDAR scan. Intermediate fusion allows the network to combine these spatial features while they are still in their 2D representation, leading to a richer joint representation.

Among the intermediate strategies, Multiplication outperformed Concating (F1 0.85) and addition (F1 0.81). The relationship between the modalities is best modeled as a gating mechanism. For instance, the LiDAR depth data might effectively "mask" irrelevant data in the RGB image by multiplying by 0. Addition, likely strengthened the noise signal, while Concatenation increased the dimensionality without forcing immediate interaction, leading to good but not optimal performance.

Overall, the Multiplying Intermediate Fusion architecture is the most efficient and effective design for this task. We validated the theoretical benefit of intermediate fusion and showed that a smaller model can outperform a larger model.